In [5]:
from pathlib import Path
import shutil
import subprocess

EXCERPT_ID = "excerpt_000"

# Notebook location:
# storyfx/pre_processing/MFA/alignment.ipynb

PROJECT_ROOT = Path.cwd().parents[1]

# ============================================================
# DATA DIRECTORIES
# ============================================================

DATA_DIR = PROJECT_ROOT / "data"
EXCERPT_DIR = DATA_DIR / EXCERPT_ID

RAW_DIR = EXCERPT_DIR / "raw"
ALIGNMENT_DIR = EXCERPT_DIR / "alignment"

# ============================================================
# MFA WORKING DIRECTORIES
# ============================================================

MFA_DIR = PROJECT_ROOT / "scripts" / "MFA"
CORPUS_DIR = MFA_DIR / "corpus"
MFA_OUTPUT_DIR = MFA_DIR / "aligned_output"

# ============================================================
# RAW INPUT FILES
# ============================================================

AUDIO_FILE = RAW_DIR / f"{EXCERPT_ID}.wav"
PRAAT_TEXTGRID = RAW_DIR / f"{EXCERPT_ID}.textgrid"

# ============================================================
# FINAL OUTPUT FILES
# ============================================================

FINAL_TEXTGRID = ALIGNMENT_DIR / f"{EXCERPT_ID}_fa.TextGrid"
FINAL_PARSED = ALIGNMENT_DIR / f"{EXCERPT_ID}_fa_parsed.txt"

# ============================================================
# PRINT PATHS
# ============================================================

print("Project root:", PROJECT_ROOT)
print("Excerpt:", EXCERPT_ID)

print("\nMFA inputs:")
print("Audio:", AUDIO_FILE)
print("Praat TextGrid:", PRAAT_TEXTGRID)

print("\nMFA working directories:")
print("Corpus:", CORPUS_DIR)
print("Temporary output:", MFA_OUTPUT_DIR)

print("\nFinal outputs:")
print("Aligned TextGrid:", FINAL_TEXTGRID)
print("Parsed FA:", FINAL_PARSED)

Project root: C:\Users\gorda\Desktop\StoryFX\storyfx
Excerpt: excerpt_000

MFA inputs:
Audio: C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\raw\excerpt_000.wav
Praat TextGrid: C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\raw\excerpt_000.textgrid

MFA working directories:
Corpus: C:\Users\gorda\Desktop\StoryFX\storyfx\scripts\MFA\corpus
Temporary output: C:\Users\gorda\Desktop\StoryFX\storyfx\scripts\MFA\aligned_output

Final outputs:
Aligned TextGrid: C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\alignment\excerpt_000_fa.TextGrid
Parsed FA: C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\alignment\excerpt_000_fa_parsed.txt


In [3]:
subprocess.run(
    [
        "micromamba",
        "create",
        "-y",
        "-n",
        "mfa_env",
        "python=3.10"
    ],
    check=True
)

print("Created mfa_env.")

Created mfa_env.


In [4]:
subprocess.run(
    [
        "micromamba",
        "install",
        "-y",
        "-n",
        "mfa_env",
        "-c",
        "conda-forge",
        "montreal-forced-aligner"
    ],
    check=True
)

print("MFA installed.")

CompletedProcess(args=['micromamba', 'install', '-y', '-n', 'mfa_env', '-c', 'conda-forge', 'montreal-forced-aligner'], returncode=0)

In [6]:
subprocess.run(
    [
        "micromamba",
        "run",
        "-n",
        "mfa_env",
        "mfa",
        "model",
        "download",
        "acoustic",
        "english_us_arpa"
    ],
    check=True
)

subprocess.run(
    [
        "micromamba",
        "run",
        "-n",
        "mfa_env",
        "mfa",
        "model",
        "download",
        "dictionary",
        "english_us_arpa"
    ],
    check=True
)

print("MFA models downloaded.")

MFA models downloaded.


In [7]:
# Validate input files

if not AUDIO_FILE.exists():
    raise FileNotFoundError(
        f"Missing WAV file: {AUDIO_FILE}"
    )

if not PRAAT_TEXTGRID.exists():
    raise FileNotFoundError(
        f"Missing Praat TextGrid: {PRAAT_TEXTGRID}"
    )


# Clean temporary MFA directories

if CORPUS_DIR.exists():
    shutil.rmtree(CORPUS_DIR)

if MFA_OUTPUT_DIR.exists():
    shutil.rmtree(MFA_OUTPUT_DIR)


# Recreate directories

CORPUS_DIR.mkdir(parents=True, exist_ok=True)
MFA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALIGNMENT_DIR.mkdir(parents=True, exist_ok=True)


# Copy WAV + manually segmented Praat TextGrid into MFA corpus

shutil.copy2(
    AUDIO_FILE,
    CORPUS_DIR / f"{EXCERPT_ID}.wav"
)

shutil.copy2(
    PRAAT_TEXTGRID,
    CORPUS_DIR / f"{EXCERPT_ID}.TextGrid"
)


print("MFA corpus prepared successfully.")

print(
    CORPUS_DIR / f"{EXCERPT_ID}.wav"
)

print(
    CORPUS_DIR / f"{EXCERPT_ID}.TextGrid"
)

MFA corpus prepared successfully.
C:\Users\gorda\Desktop\StoryFX\storyfx\scripts\MFA\corpus\excerpt_000.wav
C:\Users\gorda\Desktop\StoryFX\storyfx\scripts\MFA\corpus\excerpt_000.TextGrid


In [8]:
subprocess.run(
    [
        "micromamba",
        "run",
        "-n",
        "mfa_env",
        "mfa",
        "align",
        str(CORPUS_DIR),
        "english_us_arpa",
        "english_us_arpa",
        str(MFA_OUTPUT_DIR),
        "--clean"
    ],
    check=True
)

print("MFA alignment complete.")

MFA alignment complete.


In [9]:
MFA_TEXTGRID = MFA_OUTPUT_DIR / f"{EXCERPT_ID}.TextGrid"

if not MFA_TEXTGRID.exists():
    raise FileNotFoundError(
        f"Expected MFA output not found: {MFA_TEXTGRID}"
    )

shutil.copy2(
    MFA_TEXTGRID,
    FINAL_TEXTGRID
)

print("Saved:", FINAL_TEXTGRID)

Saved: C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\alignment\excerpt_000_fa.TextGrid


In [11]:
!pip install praatio

In [12]:
from praatio import textgrid

tg = textgrid.openTextgrid(
    str(FINAL_TEXTGRID),
    includeEmptyIntervals=False
)

word_tier = tg.getTier("words")

parsed_words = []

for entry in word_tier.entries:
    start, end, word = entry

    word = word.strip()

    if word:
        parsed_words.append(
            f"{word}|{round(end, 2)}"
        )

parsed_text = " ".join(parsed_words)

print(parsed_text)

why|0.93 they|1.07 came|1.38 east|1.66 i|1.77 don't|1.94 know|2.36 they|3.33 had|3.46 spent|3.68 a|3.71 year|3.92 in|4.07 france|4.6 for|4.76 no|4.96 particular|5.54 reason|6.02 and|6.57 then|6.79 drifted|7.19 here|7.44 and|7.65 there|7.92 unrestfully|8.76 wherever|9.47 people|9.81 played|10.25 polo|10.69 and|10.81 were|10.98 rich|11.32 together|11.83 this|12.36 was|13.98 a|14.04 permanent|14.62 move|15.04 said|15.75 daisy|16.09 over|16.29 the|16.38 telephone|16.98 but|17.61 i|17.75 didn't|18.02 believe|18.39 it|18.59 i|19.61 had|19.79 no|19.99 sight|20.33 into|20.53 daisy's|20.92 heart|21.29 but|21.91 i|22.03 felt|22.25 that|22.44 tom|22.75 would|22.91 drift|23.26 on|23.47 forever|23.96 seeking|24.48 a|24.56 little|24.85 wistfully|25.54 for|25.7 the|25.79 dramatic|26.42 turbulence|27.07 of|27.2 some|27.46 irrecoverable|28.3 football|28.8 game|29.24 and|30.65 so|30.9 it|31.0 happened|31.53 that|31.72 on|31.89 a|31.96 warm|32.41 windy|32.9 evening|33.37 i|34.01 drove|34.36 over|34.6 to|

In [13]:
with open(
    FINAL_PARSED,
    "w",
    encoding="utf-8"
) as f:
    f.write(parsed_text)

print("Saved:", FINAL_PARSED)

Saved: C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\alignment\excerpt_000_fa_parsed.txt
